# Sliding Window Retraining for Fraud Detection

Compares **static** (train on all past data) vs **sliding window** (train on fixed recent window only) retraining strategies under temporal evaluation.

## 1. Imports and Config

In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import lightgbm as lgb

# Add project root for imports
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if not os.path.exists(os.path.join(ROOT, "src")):
    ROOT = os.path.dirname(ROOT)  # fallback if run from project root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.features_aligned import prepare_features_aligned
from src.eval import pr_auc, recall_at_fpr

# ----------------------------
# CONFIG
# ----------------------------
TARGET_COL = "isFraud"
TIME_COL = "TransactionDT"

# Data path (try parquet first, fallback to csv)
DATA_DIR = os.path.join(ROOT, "data")
if os.path.exists(os.path.join(DATA_DIR, "train_merged.parquet")):
    DATA_PATH = os.path.join(DATA_DIR, "train_merged.parquet")
    load_fn = lambda: pd.read_parquet(DATA_PATH)
elif os.path.exists(os.path.join(DATA_DIR, "train_transaction.csv")):
    DATA_PATH = os.path.join(DATA_DIR, "train_transaction.csv")
    load_fn = lambda: pd.read_csv(DATA_PATH)
else:
    raise FileNotFoundError("Neither train_merged.parquet nor train_transaction.csv found in data/") 

# Sliding window: number of folds and window size (as fraction of data per chunk)
N_FOLDS = 5
WINDOW_CHUNKS = 1  # Train on last 1 chunk only (true sliding)

# LightGBM hyperparameters (match src/train.py)
LGB_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 200,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary",
    "n_jobs": -1,
    "random_state": 42,
}

# Results
RESULTS_DIR = os.path.join(ROOT, "results")
RESULTS_CSV = os.path.join(RESULTS_DIR, "sliding_window_results.csv")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Data: {DATA_PATH}")
print(f"Results: {RESULTS_CSV}")

## 2. Load Data and Define Splits

In [ ]:
df = load_fn()
df = df.sort_values(TIME_COL).reset_index(drop=True)

assert TARGET_COL in df.columns, f"Missing {TARGET_COL}"
assert TIME_COL in df.columns, f"Missing {TIME_COL}"

n = len(df)
n_chunks = N_FOLDS + 1
edges = np.linspace(0, n, num=n_chunks + 1, dtype=int)

chunk_size = edges[1] - edges[0]
window_rows = chunk_size * WINDOW_CHUNKS

print(f"Total rows: {n}")
print(f"Chunk size: {chunk_size}")
print(f"Sliding window: {window_rows} rows ({WINDOW_CHUNKS} chunk(s))")
print(f"Folds: {N_FOLDS}")

## 3. Train and Evaluate Function

In [ ]:
def train_and_eval_aligned(train_df, val_df):
    """Train LightGBM with aligned encoding, return metrics."""
    X_train, y_train, X_val, y_val = prepare_features_aligned(train_df, val_df)
    
    n_pos = int((y_train == 1).sum())
    if n_pos < 10:
        return {"pr_auc": np.nan, "recall@1%fpr": np.nan, "error": "Too few positives"}
    
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    
    return {
        "pr_auc": float(pr_auc(y_val, y_pred)),
        "recall@1%fpr": float(recall_at_fpr(y_val, y_pred, 0.01)),
    }

## 4. Run Sliding vs Static Comparison

In [ ]:
rows = []

for fold in range(N_FOLDS):
    va_start, va_end = edges[fold + 1], edges[fold + 2]
    val_df = df.iloc[va_start:va_end].reset_index(drop=True)
    
    if len(val_df) == 0:
        continue
    
    # Sliding: train on last window_rows before val
    sw_end = va_start
    sw_start = max(0, sw_end - window_rows)
    train_sliding = df.iloc[sw_start:sw_end].reset_index(drop=True)
    
    # Static: train on all past
    train_static = df.iloc[0:va_start].reset_index(drop=True)
    
    if len(train_static) == 0:
        continue
    
    print(f"\n===== Fold {fold} =====")
    print(f"  Val: {len(val_df)} rows")
    print(f"  Static train: {len(train_static)} rows")
    print(f"  Sliding train: {len(train_sliding)} rows")
    
    res_static = train_and_eval_aligned(train_static, val_df)
    res_sliding = train_and_eval_aligned(train_sliding, val_df)
    
    print(f"  Static  PR-AUC: {res_static['pr_auc']:.4f}  Recall@1%FPR: {res_static['recall@1%fpr']:.4f}")
    print(f"  Sliding PR-AUC: {res_sliding['pr_auc']:.4f}  Recall@1%FPR: {res_sliding['recall@1%fpr']:.4f}")
    
    rows.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "fold": fold,
        "strategy": "static",
        "train_rows": len(train_static),
        "val_rows": len(val_df),
        "pr_auc": res_static["pr_auc"],
        "recall_at_1pct_fpr": res_static["recall@1%fpr"],
    })
    rows.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "fold": fold,
        "strategy": "sliding",
        "train_rows": len(train_sliding),
        "val_rows": len(val_df),
        "pr_auc": res_sliding["pr_auc"],
        "recall_at_1pct_fpr": res_sliding["recall@1%fpr"],
    })

results_df = pd.DataFrame(rows)

## 5. Save and Display Results

In [ ]:
results_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved to {RESULTS_CSV}")

print("\n--- Summary by strategy ---")
summary = results_df.groupby("strategy").agg({
    "pr_auc": ["mean", "std"],
    "recall_at_1pct_fpr": ["mean", "std"],
}).round(4)
print(summary)

print("\n--- Full results ---")
print(results_df.to_string())